In [0]:
from pyspark.sql import functions as F

SILVER_PATH = "/Volumes/workspace/default/insure_data/silver"
GOLD_PATH = "/Volumes/workspace/default/insure_data/gold"

agent_df = spark.read.format("delta").load(f"{SILVER_PATH}/agent")
customer_df = spark.read.format("delta").load(f"{SILVER_PATH}/customer")
policy_df = spark.read.format("delta").load(f"{SILVER_PATH}/policy")
agent_policy_df = spark.read.format("delta").load(f"{SILVER_PATH}/agent_policy")
customer_policy_df = spark.read.format("delta").load(f"{SILVER_PATH}/customer_policy")
money_in_df = spark.read.format("delta").load(f"{SILVER_PATH}/money_in_dtl")
product_df = spark.read.format("delta").load(f"{SILVER_PATH}/product_master")
commission_df = spark.read.format("delta").load(f"{SILVER_PATH}/product_commission_rule")

In [0]:
premium_df = (
    money_in_df
    .filter(F.col("payment_status") == "SUCCESS")
    .groupBy("policy_no")
    .agg(
        F.sum("premium_amount").alias("total_premium_till_date")
    )
)

In [0]:
#join tables
gold_df = (
    agent_policy_df.alias("ap")

    .join(
        agent_df.alias("a"),
        "agent_no",
        "inner"
    )

    .join(
        policy_df.alias("p"),
        "policy_no",
        "inner"
    )

    .join(
        customer_policy_df.alias("cp"),
        "policy_no",
        "inner"
    )

    .join(
        customer_df.alias("c"),
        "customer_no",
        "inner"
    )

    .join(
        premium_df.alias("pm"),
        "policy_no",
        "left"
    )

    .join(
        product_df.alias("prd"),
        "product_id",
        "left"
    )

    .join(
        commission_df.alias("cr"),
        "product_id",
        "left"
    )
)

In [0]:
# commission calculation
gold_df = gold_df.withColumn(
    "agent_commission",
    F.round(
        F.col("total_premium_till_date") *
        F.col("commission_percentage") / 100,
        2
    )
)

In [0]:
#seelct Final columns
gold_df = gold_df.select(

    F.col("agent_no").alias("agent_id"),

    "policy_no",

    "agent_full_name",

    F.col("city").alias("agent_location"),

    "customer_no",

    F.col("cust_full_name").alias("customer_full_name"),

    "sum_assured",

    "total_premium_till_date",

    "agent_commission"
)

In [0]:
# Show data

gold_df.show(10, False)

print(
    "Gold Records :",
    gold_df.count()
)

In [0]:
#--to save in delta table format 
'''
(
    gold_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{GOLD_PATH}/agent_incentive")
)


print("Gold Delta table created successfully.")
'''
#--to save in CSV format
CSV_PATH = "/Volumes/workspace/default/insure_data/gold/agent_commision"
(
    gold_df.coalesce(1)      # Creates a single CSV file
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(CSV_PATH)
)
print("Gold CSV created successfully.")

In [0]:
# rename csv

files = dbutils.fs.ls("/Volumes/workspace/default/insure_data/gold/agent_commision/")

for file in files:
    if file.name.endswith(".csv"):
        dbutils.fs.mv(
            file.path,
            "/Volumes/workspace/default/insure_data/gold/agent_incentive.csv"
        )